# Archived

This notebook has been refactored into:
- `main.py` — data pipeline
- `index.html` + `style.css` + `script.js` — interactive map

Run `uv run main.py` to regenerate the CSV, then serve with `python -m http.server` and open `index.html`.

In [ ]:
import requests
import geopandas as gpd
from io import StringIO
import osm2geojson

In [ ]:


query = """
[out:json][timeout:300];
area["ISO3166-2"="GB-SCT"]->.scotland;
(
  way["natural"="water"](area.scotland);
  relation["natural"="water"](area.scotland);
);
out geom;
"""

headers = {
    "User-Agent": "ortom-tarn-finder/0.1 (tom@ortom.ai)",
    "Accept": "application/json",
}

r = requests.post(
    "https://overpass-api.de/api/interpreter",
    data={"data": query},
    headers=headers,
    timeout=360,
)
r.raise_for_status()
data = r.json()


In [ ]:
geojson = osm2geojson.json2geojson(data)
gdf = gpd.GeoDataFrame.from_features(geojson["features"], crs="EPSG:4326")

In [ ]:
# Fix invalid geometries first
gdf["geometry"] = gdf.geometry.buffer(0)
gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty]

# Project to British National Grid for area in m²
gdf = gdf.to_crs("EPSG:27700")
gdf["area_m2"] = gdf.geometry.area

# Size filter:
gdf = gdf[gdf["area_m2"] >= 5000]

# Exclude manmade water
gdf["water_type"] = gdf["tags"].apply(lambda t: (t or {}).get("water"))
gdf["name"] = gdf["tags"].apply(lambda t: (t or {}).get("name"))
gdf = gdf[~gdf["water_type"].isin([
    "reservoir", "basin", "wastewater", "lagoon",
    "river", "canal", "stream", "drain", "moat", "lock", "stream_pool"
])]

print(f"After size + type filter: {len(gdf)}")
print(gdf["water_type"].value_counts(dropna=False))

In [ ]:
import rasterio
import numpy as np

with rasterio.open("scotland_terrain50.vrt") as src:
    centroids = gdf.geometry.centroid
    coords = [(pt.x, pt.y) for pt in centroids]
    elevations = np.array([v[0] for v in src.sample(coords)])

gdf["elevation_m"] = elevations

# Step 4 — altitude filter
high_water = gdf[gdf["elevation_m"] >= 400].copy()
high_water = high_water.sort_values("elevation_m", ascending=False)



In [ ]:

print(f"Bodies of water 5000m2 diameter and ≥400m altitude: {len(high_water)}")
print(high_water[["name", "water_type", "area_m2", "elevation_m"]].head(30))

In [ ]:
len(gdf[(gdf["elevation_m"] >= 500) & (gdf["area_m2"] > 7000)])

In [ ]:
#dedupe
high_water["centroid"] = high_water.geometry.centroid
high_water["cx"] = (high_water.centroid.x / 50).round() * 50  # snap to 50m grid
high_water["cy"] = (high_water.centroid.y / 50).round() * 50
high_water = high_water.drop_duplicates(subset=["cx", "cy"])
print(f"After deduplication: {len(high_water)}")

In [ ]:
from shapely.geometry import Point
from scipy.spatial.distance import pdist
import numpy as np

def max_diameter(geom):
    """Longest straight line across a polygon, in the geometry's CRS units."""
    if geom is None or geom.is_empty:
        return np.nan
    hull = geom.convex_hull
    # Handle both Polygon and MultiPolygon
    if hull.geom_type == "Polygon":
        coords = np.array(hull.exterior.coords)
    else:
        # Degenerate case, fall back to bounds diagonal
        minx, miny, maxx, maxy = geom.bounds
        return np.hypot(maxx - minx, maxy - miny)
    # pdist gives pairwise distances; max is the diameter
    return float(pdist(coords).max())

# Apply — gdf is in EPSG:27700 so units are metres
gdf["length_m"] = gdf.geometry.apply(max_diameter).round().astype(int)

In [ ]:
# Your high_water GeoDataFrame should be in EPSG:27700 (British National Grid)
# If not: high_water = high_water.to_crs("EPSG:27700")

df = high_water.copy()
df["length_m"] = df.geometry.apply(max_diameter).round().astype(int)

# BNG coordinates for streetmap
df["easting"] = df.geometry.centroid.x.astype(int)
df["northing"] = df.geometry.centroid.y.astype(int)

# WGS84 for Google Maps (useful fallback)
wgs = df.geometry.centroid.to_crs("EPSG:4326")
df["lat"] = wgs.y.round(5)
df["lon"] = wgs.x.round(5)

# Build clickable links
df["os_map"] = (
    "https://www.streetmap.co.uk/map.srf?X=" + df["easting"].astype(str) +
    "&Y=" + df["northing"].astype(str) + "&Z=115&A=Y"
)
df["google_sat"] = (
    "https://www.google.com/maps/@?api=1&map_action=map&center=" +
    df["lat"].astype(str) + "," + df["lon"].astype(str) +
    "&zoom=15&basemap=satellite"
)


In [ ]:
def compactness(geom):
    """Ratio of polygon area to its oriented bounding box area.
    1.0 = perfectly rectangular/filled, ~0.1 = long thin river."""
    if geom is None or geom.is_empty:
        return np.nan
    rect = geom.minimum_rotated_rectangle
    if rect.area == 0:
        return np.nan
    return geom.area / rect.area

df["compactness"] = df.geometry.apply(compactness).round(3)

# Also compute aspect ratio — length:width of bounding box
def aspect_ratio(geom):
    if geom is None or geom.is_empty:
        return np.nan
    rect = geom.minimum_rotated_rectangle
    coords = np.array(rect.exterior.coords)
    sides = np.linalg.norm(np.diff(coords, axis=0), axis=1)
    long, short = sides.max(), sides.min()
    return float(long / short) if short > 0 else np.nan

df["aspect_ratio"] = df.geometry.apply(aspect_ratio).round(2)

In [ ]:
#dont ttruncate in print for geodf
import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)  # don't wrap
pd.set_option("display.max_columns", None)  # show all columns



print(df[["name", "length_m", "compactness", "aspect_ratio", "os_map"]]
      .sort_values("aspect_ratio", ascending=False).head(20))

In [ ]:

# Clean up and order columns
out = df[[
    "name", "water_type", "area_m2", "elevation_m","length_m",
    "lat", "lon", "os_map", "google_sat", "compactness", "aspect_ratio"
]].rename(columns={
    "area_m2": "area_sqm",
    "elevation_m": "elevation",
})

# Round area, tidy up
out["area_sqm"] = out["area_sqm"].round().astype(int)
out["elevation"] = out["elevation"].round(1)
out["area_hectares"] = (out["area_sqm"] / 10000).round(2)
out["name"] = out["name"].fillna("(unnamed)")
out = out.sort_values("elevation", ascending=False)

out.to_csv("scottish_high_lochs.csv", index=False)

In [ ]:
import folium
import geopandas as gpd
import pandas as pd

OS_API_KEY = "vt6TzPGU43fdeK6Pjg52sbWXshKSrxB5"

# Pre-filter: exclude river-like shapes
hw_filtered = df[df["aspect_ratio"] <= 5].copy()
print(f"After aspect ratio filter: {len(hw_filtered)} (removed {len(df) - len(hw_filtered)})")

# Build points GeoDataFrame
hw_points = gpd.GeoDataFrame(
    {
        "name": hw_filtered["name"].fillna("(unnamed)"),
        "elevation_m": hw_filtered["elevation_m"].round().astype(int),
        "area_ha": (hw_filtered["area_m2"] / 10000).round(2),
        "length_m": hw_filtered["length_m"].astype(int),
        "water_type": hw_filtered["water_type"].fillna("water"),
    },
    geometry=hw_filtered.geometry.centroid,
    crs=hw_filtered.crs,
).to_crs("EPSG:4326")

# Slider ranges
elev_min, elev_max = int(hw_points.elevation_m.min()), int(hw_points.elevation_m.max())
len_min, len_max = int(hw_points.length_m.min()), int(hw_points.length_m.max())

m = folium.Map(location=[57.1, -3.7], zoom_start=8)

folium.TileLayer(
    tiles=f"https://api.os.uk/maps/raster/v1/zxy/Outdoor_3857/{{z}}/{{x}}/{{y}}.png?key={OS_API_KEY}",
    attr="Contains OS data © Crown copyright and database right 2026",
    name="OS Outdoor",
    max_zoom=20,
).add_to(m)

geojson_layer = folium.GeoJson(
    hw_points,
    name="lochs",
    marker=folium.CircleMarker(radius=5, fill=True, fill_opacity=0.8, color="blue"),
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "elevation_m", "area_ha", "length_m"],
        aliases=["Name", "Elevation (m)", "Area (ha)", "Length (m)"],
    ),
).add_to(m)

folium.LayerControl().add_to(m)

filter_html = f"""
<div id="filter-panel" style="
    position: fixed; top: 10px; right: 10px; z-index: 1000;
    background: white; padding: 12px; border-radius: 6px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.3); font-family: sans-serif;
    font-size: 13px; width: 260px;">
  <b>Filters</b>
  <div style="margin-top: 8px;">
    Min elevation: <span id="elev-val">{elev_min}</span> m<br>
    <input type="range" id="elev-min" min="{elev_min}" max="{elev_max}"
           value="{elev_min}" step="10" style="width:100%">
  </div>
  <div style="margin-top: 8px;">
    Min length: <span id="len-val">{len_min}</span> m<br>
    <input type="range" id="len-min" min="{len_min}" max="{len_max}"
           value="{len_min}" step="10" style="width:100%">
  </div>
  <div style="margin-top: 10px; font-weight: bold;">
    Showing <span id="count">0</span> of <span id="total">0</span>
  </div>
</div>

<script>
function installFilters() {{
  var layer = null;
  for (var k in window) {{
    if (k.startsWith("geo_json_") && window[k] instanceof L.GeoJSON) {{
      layer = window[k];
      break;
    }}
  }}
  if (!layer) {{ setTimeout(installFilters, 200); return; }}

  var features = [];
  layer.eachLayer(function(l) {{
    features.push({{ layer: l, props: l.feature.properties }});
  }});
  document.getElementById("total").textContent = features.length;

  function apply() {{
    var eMin = +document.getElementById("elev-min").value;
    var lMin = +document.getElementById("len-min").value;
    document.getElementById("elev-val").textContent = eMin;
    document.getElementById("len-val").textContent = lMin;

    var shown = 0;
    features.forEach(function(f) {{
      var p = f.props;
      var keep = p.elevation_m >= eMin && p.length_m >= lMin;
      if (keep) {{
        if (!layer.hasLayer(f.layer)) layer.addLayer(f.layer);
        shown++;
      }} else {{
        if (layer.hasLayer(f.layer)) layer.removeLayer(f.layer);
      }}
    }});
    document.getElementById("count").textContent = shown;
  }}

  ["elev-min", "len-min"].forEach(function(id) {{
    document.getElementById(id).addEventListener("input", apply);
  }});
  apply();
}}

document.addEventListener("DOMContentLoaded", installFilters);
</script>
"""

m.get_root().html.add_child(folium.Element(filter_html))
m.save("scottish_high_lochs.html")
print("Saved to scottish_high_lochs.html")